# Week 7 — CNNs for Melanoma Classification (Sprint 2)
## Melanoma Skin Cancer Classification: Benign vs Malignant

**BinX Tech — AI & Machine Learning Internship Program | Phase 3, Sprint 2**

This notebook advances the Phase 3 core model from Week 6. Sprint 1 proved that a
plain Dense network cannot beat a simple baseline on raw image pixels. This sprint
fixes that by using the architecture that actually fits image data: a CNN.

## Sprint 2 Planning

**Sprint Goal:** Build a CNN-based core model that beats the Week 6 baseline
(Logistic Regression, 86.85% accuracy) and the best Week 6 neural network (NN v3,
82.60%), by using an architecture that respects the spatial structure of images.

**Retrospective improvement carried forward from Sprint 1:** change one
hyperparameter at a time per experiment, instead of changing two at once (like the
learning rate + dropout change between v2 and v3 last sprint).

**Backlog for this sprint:**
- [ ] Reload the dataset WITHOUT flattening (keep spatial structure)
- [ ] Explain why the Week 6 Dense network failed
- [ ] Hand-coded convolution demo (edge detection filter + feature map)
- [ ] Build a CNN from scratch, train, evaluate (v1)
- [ ] Add data augmentation, retrain (v2)
- [ ] Apply transfer learning with MobileNetV2 (v3)
- [ ] Compare all models (baseline, Week 6 NNs, CNN v1/v2/v3) in one table
- [ ] Sprint Review and Retrospective

**Definition of done:** Notebook runs top to bottom without errors, every model's
score is recorded and compared to the baseline, and the final architecture choice
is justified in Markdown.

In [ ]:
# import data from kaggle

from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!mkdir -p ~/.kaggle
!echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!pip install kaggle -q
!kaggle datasets download -d ailearner-researchlab/melanoma-skin-cancer-dataset-benign-vs-malignant
!unzip -q melanoma-skin-cancer-dataset-benign-vs-malignant.zip -d melanoma_data

## 0. Why the Week 6 Dense Network Failed

Last sprint I flattened every image into a 1D array of 12,288 numbers before
feeding it to a Dense network:

X_train_flat = X_train_norm.reshape(X_train_norm.shape[0],-1)


This threw away all spatial information — the Dense network had no way to know
that pixel 500 and pixel 501 might be neighbors in the original image. It treated
every pixel as an independent feature, exactly like a tabular dataset. That's why
none of the three neural network versions (70.70%, 82.20%, 82.60%) beat the
86.85% Logistic Regression baseline.

A CNN fixes this by keeping the image's 2D shape and sliding small filters across
it, so it can actually learn spatial patterns like edges, curves, and textures —
which is exactly what distinguishes benign from malignant lesions.

In [ ]:
# Reload the same dataset as Week 6, but this time DO NOT flatten it.
# We reuse the loading logic from Week 6's Sprint1.ipynb.

import numpy as np
from PIL import Image
import os

IMG_SIZE = 64

def load_images_from_folder(folder_path, label, img_size=IMG_SIZE):
    images = []
    labels = []
    files = os.listdir(folder_path)
    for fname in files:
        try:
            img = Image.open(os.path.join(folder_path, fname)).convert('RGB')
            img = img.resize((img_size, img_size))
            images.append(np.array(img))
            labels.append(label)
        except Exception as e:
            print(f'Problem with image {fname}: {e}')
    return images, labels

train_benign_imgs, train_benign_labels = load_images_from_folder('melanoma_data/train/Benign', 0)
train_malignant_imgs, train_malignant_labels = load_images_from_folder('melanoma_data/train/Malignant', 1)

X_train = np.array(train_benign_imgs + train_malignant_imgs)
y_train = np.array(train_benign_labels + train_malignant_labels)

test_benign_imgs, test_benign_labels = load_images_from_folder('melanoma_data/test/Benign', 0)
test_malignant_imgs, test_malignant_labels = load_images_from_folder('melanoma_data/test/Malignant', 1)

X_test = np.array(test_benign_imgs + test_malignant_imgs)
y_test = np.array(test_benign_labels + test_malignant_labels)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

In [ ]:
from sklearn.utils import shuffle

# Normalize pixel values to 0-1
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# Shuffle (data was loaded in order: all Benign, then all Malignant)
X_train_norm, y_train = shuffle(X_train_norm, y_train, random_state=42)
X_test_norm, y_test = shuffle(X_test_norm, y_test, random_state=42)

# CRITICAL DIFFERENCE FROM WEEK 6: we do NOT reshape/flatten here.
# Shape stays (n_samples, 64, 64, 3) so the CNN can see spatial structure.
print(f'X_train_norm shape (kept 2D): {X_train_norm.shape}')
print(f'X_test_norm shape (kept 2D): {X_test_norm.shape}')

## 1. Understanding Convolution Before Building the CNN

Before building the full network, I want to see convolution actually working on
one of my own melanoma images — the same way I manually coded a forward pass in
Week 6 before trusting `model.fit()`.

In [ ]:
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

# Take one sample image, convert to grayscale for a simple demo
sample_img = X_train_norm[0]
gray_img = np.mean(sample_img, axis=2)  # simple grayscale conversion

# A classic vertical edge detection filter (Sobel-style)
edge_filter = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

feature_map = convolve2d(gray_img, edge_filter, mode='same')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample_img)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(gray_img, cmap='gray')
axes[1].set_title('Grayscale')
axes[1].axis('off')

axes[2].imshow(feature_map, cmap='gray')
axes[2].set_title('Feature Map (Edge Filter)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

**Result:** the edge filter highlights boundaries in the lesion — exactly where
pixel intensity changes sharply (the edge of a mole, a color transition). This
one 3x3 filter (9 weights) was reused across the *entire* image to produce this
feature map. A Dense layer doing the same job on a 64x64 image would need a
separate weight for every single pixel position — thousands of times more
parameters for the same job. This is parameter sharing, and it's the core reason
CNNs are efficient on images.

**Data type confirmation:** my project is image data (Melanoma Benign vs
Malignant), so per the Week 7 architecture-matching table, the correct core
model is a **CNN**, with transfer learning as the recommended approach for
strong results on a relatively small medical imaging dataset.

## 2. Building a CNN From Scratch (v1)

Now I'll build my first real CNN. Unlike Week 6's Dense network, this one keeps
the image's 2D shape throughout the convolution/pooling stages, and only
flattens at the very end — after the spatial features have already been
extracted, not before.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

cnn_v1 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_v1.summary()

In [ ]:
cnn_v1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_cnn_v1 = cnn_v1.fit(
    X_train_norm, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cnn_v1.history['loss'], label='Training Loss')
axes[0].plot(history_cnn_v1.history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss - CNN v1 (from scratch)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_cnn_v1.history['accuracy'], label='Training Accuracy')
axes[1].plot(history_cnn_v1.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Accuracy - CNN v1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_v1_curves.png', dpi=150)
plt.show()

In [ ]:
test_loss_cnn1, test_acc_cnn1 = cnn_v1.evaluate(X_test_norm, y_test)

print(f'\n=== Comparison So Far ===')
print(f'Baseline (Logistic Regression - Week 6):        86.85%')
print(f'NN v3 (best Dense network - Week 6):             82.60%')
print(f'CNN v1 (from scratch, no augmentation):          {test_acc_cnn1*100:.2f}%')

**Diagnosis:** [Fill in after running — e.g. "CNN v1 scored X%, which is/isn't
better than the Week 6 baseline. Looking at the loss curves, ..."]

## 3. Adding Data Augmentation (v2)

My melanoma dataset is relatively small for a CNN. Data augmentation artificially
expands it by showing the network randomly flipped, rotated, and zoomed versions
of the same images — forcing it to generalize instead of memorize.

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom

data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),
])

cnn_v2 = Sequential([
    data_augmentation,
    Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_v2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_cnn_v2 = cnn_v2.fit(
    X_train_norm, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cnn_v2.history['loss'], label='Training Loss')
axes[0].plot(history_cnn_v2.history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss - CNN v2 (with augmentation)')
axes[0].legend()

axes[1].plot(history_cnn_v2.history['accuracy'], label='Training Accuracy')
axes[1].plot(history_cnn_v2.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Accuracy - CNN v2')
axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_v2_curves.png', dpi=150)
plt.show()

test_loss_cnn2, test_acc_cnn2 = cnn_v2.evaluate(X_test_norm, y_test)

print(f'\n=== Comparison So Far ===')
print(f'Baseline (Logistic Regression - Week 6):    86.85%')
print(f'CNN v1 (from scratch, no augmentation):      {test_acc_cnn1*100:.2f}%')
print(f'CNN v2 (from scratch + augmentation):        {test_acc_cnn2*100:.2f}%')

**Diagnosis:** [Fill in after running — did augmentation help close the gap with
the baseline, or not? Why might that be, given dataset size?]

## 4. Transfer Learning with MobileNetV2 (v3)

Training a CNN from scratch needs a lot of data to learn good features.
Transfer learning reuses a model already trained on millions of images
(MobileNetV2 on ImageNet), keeping its learned feature extractor and only
training a new classification head on top. This is the recommended approach
for an image classifier project with a moderate-sized medical dataset.

**Important:** MobileNetV2 expects a minimum input size (commonly 96x96 or
larger) and its own specific preprocessing, so I resize images up from 64x64
and use its matching `preprocess_input` function.

In [ ]:
import tensorflow as tf

MOBILENET_SIZE = 96

X_train_resized = tf.image.resize(X_train_norm, (MOBILENET_SIZE, MOBILENET_SIZE)).numpy()
X_test_resized = tf.image.resize(X_test_norm, (MOBILENET_SIZE, MOBILENET_SIZE)).numpy()

print(f'X_train_resized shape: {X_train_resized.shape}')

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D

# Note: our images are already scaled 0-1, MobileNetV2's preprocess_input
# expects 0-255 range, so we reverse the earlier /255 scaling for this branch.
X_train_mobilenet = preprocess_input(X_train_resized * 255.0)
X_test_mobilenet = preprocess_input(X_test_resized * 255.0)

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(MOBILENET_SIZE, MOBILENET_SIZE, 3)
)
base_model.trainable = False  # freeze the pre-trained feature extractor

cnn_v3 = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_v3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_v3.summary()

In [ ]:
import time

start = time.time()

history_cnn_v3 = cnn_v3.fit(
    X_train_mobilenet, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32
)

end = time.time()
print(f'\nTraining time: {end - start:.1f} seconds')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cnn_v3.history['loss'], label='Training Loss')
axes[0].plot(history_cnn_v3.history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss - CNN v3 (Transfer Learning)')
axes[0].legend()

axes[1].plot(history_cnn_v3.history['accuracy'], label='Training Accuracy')
axes[1].plot(history_cnn_v3.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Accuracy - CNN v3')
axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_v3_curves.png', dpi=150)
plt.show()

test_loss_cnn3, test_acc_cnn3 = cnn_v3.evaluate(X_test_mobilenet, y_test)
print(f'CNN v3 (Transfer Learning) Test Accuracy: {test_acc_cnn3*100:.2f}%')

**Diagnosis:** [Fill in after running — how does transfer learning compare in
both accuracy AND training time vs. the from-scratch CNNs?]

## 5. Why RNNs/LSTMs Don't Apply Here (Concept Check)

Day 3 of this sprint covers RNNs and LSTMs, which are built for **sequential**
data — text, time series, audio — where order matters and each element depends
on what came before (e.g. "not good" vs "good... not").

My melanoma project is **spatial**, not sequential: a lesion image doesn't have
a "before" and "after" in the same sense a sentence does. That's exactly why the
Week 7 architecture-matching table assigns CNNs to images and
RNNs/LSTMs/Transformers to text and sequences. I'm not building an LSTM for
this project, but understanding *why* it wouldn't fit is part of matching the
right architecture to the right data — the core skill this sprint is testing.

## 6. Why Attention/Transformers Don't Apply Here (Concept Check)

Day 4 covers the attention mechanism and Transformers, which revolutionized NLP
by letting every element in a sequence look directly at every other element
instead of processing step by step like an RNN. This solves problems specific
to sequential/text data (long-range dependencies, parallelism).

Since my project is image classification, not text, a Transformer isn't the
right tool here either — my "long-range dependency" problem doesn't exist in
the same way. (Vision Transformers exist, but for a dataset this size, CNN +
transfer learning is the more practical and appropriate choice per the Week 7
guidance.)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Model': [
        'Baseline (Logistic Regression) - Week 6',
        'NN v1 (Dense, no reg) - Week 6',
        'NN v2 (Dense + BatchNorm/Dropout) - Week 6',
        'NN v3 (Dense, tuned) - Week 6',
        'CNN v1 (from scratch) - Week 7',
        'CNN v2 (from scratch + augmentation) - Week 7',
        'CNN v3 (Transfer Learning, MobileNetV2) - Week 7'
    ],
    'Accuracy': [
        0.8685,
        0.7070,
        0.8220,
        0.8260,
        test_acc_cnn1,
        test_acc_cnn2,
        test_acc_cnn3
    ]
})

results['Accuracy (%)'] = (results['Accuracy'] * 100).round(2)
print(results[['Model', 'Accuracy (%)']].to_string(index=False))

## Sprint 2 Core Model Decision

[Fill in after seeing the table — which model are you choosing as the Phase 3
core model going forward? Justify it: did it beat the baseline? By how much?
Was transfer learning worth the trade-off vs. training time?]

**Selected core model for Sprint 3:** [state your final choice + key
hyperparameters here, e.g. "CNN v3 — MobileNetV2 transfer learning, frozen
base, GlobalAveragePooling2D + Dense(64) + Dense(1, sigmoid), Adam optimizer"]

## Sprint Review

**What was completed this sprint:**
- Diagnosed the Week 6 failure: flattening destroyed spatial structure needed
  for image classification.
- Reloaded the dataset preserving 2D image shape.
- Built and visualized a hand-coded convolution filter to confirm understanding
  of how CNNs extract spatial features.
- Built and evaluated three CNN versions:
  - v1 (from scratch): [X]%
  - v2 (from scratch + augmentation): [X]%
  - v3 (transfer learning, MobileNetV2): [X]%
- Compared all models against the Week 6 baseline (86.85%) in one table.
- Confirmed CNN is the correct architecture for image data per the Week 7
  architecture-matching guidance, and explained why RNN/LSTM/Transformer
  architectures don't apply to this project.

**Result vs. the goal:** [State honestly — did any CNN version beat the
baseline? By how much? This is the real test of the sprint goal.]

## Sprint Retrospective

**What went well:**
- [Fill in]

**What to improve:**
- [Fill in — e.g. unstable curves, need more epochs, fine-tuning the base model
  instead of freezing it entirely, etc.]

**One concrete action for Sprint 3 (Week 8):**
- Sprint 3 focuses on integration and full evaluation — build a single
  `predict()` pipeline, run proper error analysis with a confusion matrix, and
  add SHAP explainability so the model's decisions can be communicated clearly.